# 01 — Exploratory Data Analysis

This notebook explores historical OHLCV data for AAPL, MSFT, and GOOGL fetched from Yahoo Finance.  
Goals:
- Price history & cumulative returns
- Returns distribution & normality check
- Correlation heatmap
- Missing value analysis
- Technical feature distributions

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'ml-backend'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

from pipeline.ingest import fetch_stock_data, fetch_multi_stock_data
from pipeline.features import build_feature_matrix

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100
TICKERS = ['AAPL', 'MSFT', 'GOOGL']

## 1. Load Data

In [ ]:
data = fetch_multi_stock_data(TICKERS, period='2y')
for ticker, df in data.items():
    print(f'{ticker}: {len(df)} rows  |  {df["Date"].min().date()} → {df["Date"].max().date()}')
    print(df.describe().round(2), '\n')

## 2. Missing Value Analysis

In [ ]:
print('Missing values per ticker:')
for ticker, df in data.items():
    missing = df.isnull().sum()
    print(f'  {ticker}: {missing.to_dict()}')

## 3. Closing Price History

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
for ticker, df in data.items():
    ax.plot(df['Date'], df['Close'], label=ticker, linewidth=1.5)
ax.set_title('Closing Price History (2-Year)', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Price (USD)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.tight_layout()
plt.show()

## 4. Cumulative Returns

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
for ticker, df in data.items():
    returns = df.set_index('Date')['Close'].pct_change().dropna()
    cum_ret = (1 + returns).cumprod() - 1
    ax.plot(cum_ret.index, cum_ret * 100, label=ticker, linewidth=1.5)
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax.set_title('Cumulative Returns (%)', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Return (%)')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Daily Returns Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (ticker, df) in zip(axes, data.items()):
    returns = df['Close'].pct_change().dropna() * 100
    ax.hist(returns, bins=60, density=True, alpha=0.7, color='steelblue', edgecolor='none')
    # Overlay normal distribution
    mu, sigma = returns.mean(), returns.std()
    x = np.linspace(returns.min(), returns.max(), 200)
    ax.plot(x, stats.norm.pdf(x, mu, sigma), 'r-', linewidth=2, label=f'N({mu:.2f},{sigma:.2f})')
    ax.set_title(f'{ticker} Daily Returns (%)')
    ax.set_xlabel('Return (%)')
    ax.legend(fontsize=9)
    # Shapiro-Wilk test
    _, p = stats.shapiro(returns[:100])
    ax.text(0.05, 0.95, f'Shapiro p={p:.3f}', transform=ax.transAxes, fontsize=8,
            verticalalignment='top')
plt.suptitle('Daily Returns Distribution vs Normal', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Correlation Heatmap (Daily Returns)

In [ ]:
returns_dict = {}
for ticker, df in data.items():
    returns_dict[ticker] = df.set_index('Date')['Close'].pct_change()
returns_df = pd.DataFrame(returns_dict).dropna()

corr = returns_df.corr()
fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            vmin=-1, vmax=1, ax=ax, mask=mask,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Return Correlation Heatmap (2Y)', fontsize=13)
plt.tight_layout()
plt.show()
print(corr.round(4))

## 7. Technical Feature Distributions (AAPL)

In [ ]:
df_aapl = data['AAPL']
df_feat, feature_cols = build_feature_matrix(df_aapl)
print(f'Feature matrix: {df_feat.shape} | Target balance: {df_feat["target"].value_counts().to_dict()}')

# Plot feature distributions
plot_features = ['rsi', 'macd', 'bb_bandwidth', 'volume_ratio', 'close_to_sma_20', 'daily_return']
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for ax, feat in zip(axes.flat, plot_features):
    up = df_feat[df_feat['target'] == 1][feat]
    dn = df_feat[df_feat['target'] == 0][feat]
    ax.hist(up, bins=40, alpha=0.6, color='green', label='Up', density=True)
    ax.hist(dn, bins=40, alpha=0.6, color='red', label='Down', density=True)
    ax.set_title(feat)
    ax.legend(fontsize=8)
plt.suptitle('AAPL Feature Distributions by Next-Day Direction', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Volume Analysis

In [ ]:
df = data['AAPL'].copy()
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
ax1.plot(df['Date'], df['Close'], color='steelblue', linewidth=1.2)
ax1.set_ylabel('Price (USD)')
ax1.set_title('AAPL Price & Volume')
ax2.bar(df['Date'], df['Volume'] / 1e6, color='slategray', alpha=0.7, width=1)
ax2.set_ylabel('Volume (M shares)')
ax2.set_xlabel('Date')
plt.tight_layout()
plt.show()